In [3]:
import sys
import os
# Добавляем корневую директорию проекта в PYTHONPATH
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# Теперь импорт должен работать
from src.data import merge_csv_files
from src.ml_core import TopicModelEvaluator, quick_evaluate_bertopic

In [4]:
import re
from pathlib import Path
from os import path

# import emojilogging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
import logging
logger = logging.getLogger(__name__)
import warnings
warnings.filterwarnings('ignore')

# import spacy
import pandas as pd
import torch
from transformers import pipeline
from sklearn.cluster import KMeans

from umap import UMAP
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForPreTraining

from nltk.corpus import stopwords
stop_words = stopwords.words('russian')

import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords

from razdel import tokenize

from sklearn.feature_extraction.text import CountVectorizer

from keybert import KeyBERT

from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, TextGeneration
from bertopic.vectorizers import ClassTfidfTransformer  
from bertopic.dimensionality import BaseDimensionalityReduction

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vallo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [6]:
RAW_DATA_DIR = Path(r"..\data\raw").as_posix()
PROCESSED_DATA_DIR = Path(r"..\data\processed").as_posix()

In [7]:
# Объединяем все csv файлы в один
try:
    result_file = merge_csv_files(RAW_DATA_DIR, PROCESSED_DATA_DIR, logger)
    print(f"\n✅ Объединение завершено успешно!")
    print(f"📁 Результат сохранен в: {result_file}")
except Exception as e:
    logger.error(f"Критическая ошибка: {e}")
    print(f"\n❌ Ошибка при выполнении: {e}")


✅ Объединение завершено успешно!
📁 Результат сохранен в: ..\data\processed\merged_news_data.csv


In [8]:
data = pd.read_csv(result_file)
data.head()

,title,text,date,link,source,url
0,Медведчук назвал организаторов трагедии в Одес...,"Пожар у Дома профсоюзов в Одессе, 2 мая 2014 г...",2024-05-01 00:00:00,NaN,lenta-news_20240501-20240502.csv,https://lenta.ru/news/2024/05/01/medvedchuk-na...
1,Медведчук рассказал об идущем против украинцев...,Владимир Зеленский. Фото: Thomas Peter/Reuters...,2024-05-01 00:00:00,NaN,lenta-news_20240501-20240502.csv,https://lenta.ru/news/2024/05/01/medvedchuk-ra...
2,Основателя криптобиржи Binance приговорили к ч...,Чжао Чанпэн. Фото: Costas Baltas / Reuters Фед...,2024-05-01 00:00:00,NaN,lenta-news_20240501-20240502.csv,https://lenta.ru/news/2024/05/01/osnovatelya-k...
3,Президент Грузии призвала прекратить разгон пр...,Фото: Irakli Gedenidze / Reuters Президент Гру...,2024-05-01 00:00:00,NaN,lenta-news_20240501-20240502.csv,https://lenta.ru/news/2024/05/01/prezident-gru...
4,AstraZeneca признала наличие побочных эффектов...,Фото: Sergey Bulkin / Global Look Press Междун...,2024-05-01 00:00:00,NaN,lenta-news_20240501-20240502.csv,https://lenta.ru/news/2024/05/01/tromboz/


In [9]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 890 entries, 0 to 889
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   title   890 non-null    object 
 1   text    890 non-null    object 
 2   date    890 non-null    object 
 3   link    0 non-null      float64
 4   source  890 non-null    object 
 5   url     265 non-null    object 
dtypes: float64(1), object(5)
memory usage: 41.8+ KB


In [10]:
data.shape

(890, 6)

In [11]:
data.isna().sum()


title       0
text        0
date        0
link      890
source      0
url       625
dtype: int64

In [12]:
def clean_text(text):

    # Приведение текста к нижнему регистру
    # text = text.lower()s
    # text = emoji.replace_emoji(text, '')

    # Замена всех не-словесных символов на пробел (кроме букв и знаков препинания)
    text = re.sub(r'\W+', ' ', text)

    # Удаление URL-адресов
    text = re.sub(r"http\S+", "", text)

    # Создание шаблона для HTML-тегов
    html = re.compile(r'&lt;.*?&gt;')

    # Удаление HTML-тегов из текста
    text = html.sub(r'', text)

    # Список пунктуаций для удаления
    punctuations = '@#!?+&amp;*[]-%.:/();$=&gt;&lt;|{}^' + "'`" + '_'
    for p in punctuations:
        text = text.replace(p, '')  # Удаление пунктуации

    # Удаление стоп-слов и приведение слов к нижнему регистру
    text = [word for word in text.split() if word.lower() not in stop_words]

    # Объединение слов обратно в текст
    text = " ".join(text)

    # Создание шаблона для поиска эмодзи
    emoji_pattern = re.compile("["
                        u"\U0001F600-\U0001F64F"  # эмоции
                        u"\U0001F300-\U0001F5FF"  # символы и пиктограммы
                        u"\U0001F680-\U0001F6FF"  # транспорт и карты
                        u"\U0001F1E0-\U0001F1FF"  # флаги
                        u"\U00002702-\U000027B0"
                        u"\U000024C2-\U0001F251"
                        "]+", flags=re.UNICODE)

    # Удаление эмодзи из текста
    text = emoji_pattern.sub(r'', text)

    return text


In [13]:
# Создаем новый столбец с очищенным текстом
data['cleaned_text'] = data['text'].apply(clean_text)

In [14]:
data['cleaned_text'].iloc[0]

'Пожар Дома профсоюзов Одессе 2 мая 2014 года Фото Yeveny Vookin Reuers Главным организатором сожжения людей одесском Доме профсоюзов 2 мая 2014 года Александр Турчинов который исполнял обязанности президента Украины интервью ТАСС заявил бывший лидер запрещенной Украине партии Оппозиционная платформа жизнь Виктор Медведчук назвал произошедшее Одессе акцией устрашения стороны киевского режима словам 10 дней трагедии состоялось совещание котором принимали участие бывшие глава МВД Арсен Аваков глава СБУ Валентин Наливайченко секретарь Совета нацбезопасности обороны Андрей Парубий операции также привлекался Игорь Коломойский возглавлявший Днепропетровскую область Непосредственно месте отвечал Игорь Палица который руководил акцией месте успешное проведение 6 мая 2014 года назначен губернатором Одесской области должны понести суровую ответственность Злодеяния этих фашистских выродков должно иметь сроков давности подчеркнул политик официальным данным МВД Украины результате пожара Доме профсою

In [15]:
from copy import deepcopy

to_proccess_text = deepcopy(data['cleaned_text'].tolist())

In [16]:
del data

# Text classification

In [17]:
embedding_model = SentenceTransformer("deepvk/USER2-base", device=device)


KeyboardInterrupt: 

# Topic modeling

In [39]:
# embedding_model_name = "deepvk-USER2-base"
embedding_model_name = "cointegrated/rubert-tiny2"
# Load model directly
embedding_model = SentenceTransformer(embedding_model_name, device=device)
embeddings = embedding_model.encode(to_proccess_text, show_progress_bar=True)

Batches:   0%|          | 0/28 [00:00<?, ?it/s]

In [47]:
len(to_proccess_text), embeddings.shape

(890, (890, 312))

In [48]:
def tokenize_ru(text):
    words = tokenize(text)
    return [word.text for word in words]

In [49]:
dim_model = UMAP(n_neighbors=15, n_components=50, min_dist=0.0, metric='euclidean')
cluster_model = HDBSCAN(min_cluster_size=5, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
# cluster_model = KMeans(n_clusters=10, random_state=42)
vectorizer_model = CountVectorizer(tokenizer=tokenize_ru, ngram_range=(1, 2), stop_words=stop_words)
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

In [21]:

# representation_model = KeyBERTInspired()

# generator = pipeline('text2text-generation', model=representation_model_name, device=device, batch_size=100)
# representation_model = TextGeneration(generator)

In [50]:
topic_model = BERTopic(
  # language="russian",
  # embedding_model=embedding_model,          # Step 1 - Extract embeddings
  umap_model=dim_model,                     # Step 2 - Reduce dimensionality
  hdbscan_model=cluster_model,              # Step 3 - Cluster reduced embeddings
  vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
  ctfidf_model=ctfidf_model,                # Step 5 - Extract topic words
  # representation_model=representation_model # Step 6 - (Optional) Fine-tune topic represenations
)

In [51]:
topics, probs = topic_model.fit_transform(to_proccess_text, embeddings)

In [52]:
ngram_range = "3x3"

In [54]:
# Создаем более осмысленные названия для топиков
# Используем KeyBERT для генерации ключевых фраз из документов каждого топика


# Инициализируем модель KeyBERT
keybert_model = KeyBERT(model=embedding_model)

# Получаем информацию о топиках
topic_info = topic_model.get_topic_info()
topic_docs = {}

# Для каждого топика (кроме -1, который означает выбросы) получаем репрезентативные документы
for topic_id in topic_info[topic_info['Topic'] != -1]['Topic']:
    # Получаем документы для данного топика
    documents = topic_model.get_representative_docs(topic_id)
    topic_docs[topic_id] = ' '.join(documents)

# Создаем словарь для хранения новых названий топиков
topic_names = {}

# Для каждого топика генерируем ключевые фразы
for topic_id, doc in topic_docs.items():
    # Извлекаем ключевые фразы (3 слова) из документов топика
    keywords = keybert_model.extract_keywords(doc, keyphrase_ngram_range=(3, 3), stop_words=stop_words, top_n=1)
    
    if keywords:
        # Берем первую ключевую фразу как название топика
        topic_names[topic_id] = keywords[0][0]
    else:
        # Если не удалось извлечь фразу, используем оригинальное название
        words = topic_model.get_topic(topic_id)
        topic_names[topic_id] = f"Топик_{topic_id}_{words[0][0]}_{words[1][0]}"

# Переименовываем топики в модели
topic_model.set_topic_labels(topic_names)

# Выводим обновленную информацию о топиках
# print("Топики с новыми названиями:")
# display(topic_model.get_topic_info()[1:11])


In [55]:
topic_model.get_topic_info()

,Topic,Count,Name,CustomName,Representation,Representative_Docs
0,-1,156,-1_fisrs_two one_vis_one,-1_fisrs_two one_vis_one,"[fisrs, two one, vis, one, two, widberries, vi...",[смывать санскрин лица знаем гидрофильное масл...
1,0,123,0_reuers_press_look press_gob,европейские союзники настаивают,"[reuers, press, look press, gob, gob look, loo...",[Швеция прорабатывает вопрос продажи самолетов...
2,1,110,1_boxberry_2024_ikea_2025,показателей ликвидности компании,"[boxberry, 2024, ikea, 2025, 0, brin, vosok, b...",[рейтинги дайджест ДАЙДЖЕСТ РЕЙТИНГОВЫМ ДЕЙСТВ...
3,2,101,2_llm_pr_llm llm_ceo,недавно интервью эксперта,"[llm, pr, llm llm, ceo, wibes, ai, gooe, it, d...",[Сегодня Рак эндометрия вопросы клинической пр...
4,3,43,3_kinder_rocher_ferrero_rffeo,появилось мороженое вкусами,"[kinder, rocher, ferrero, rffeo, kinder bueno,...",[Привет любимые дома спасаем любимую белую обу...
5,4,33,4_widberries ru_www widberries_dei sx_ru co,ru co 349287441,"[widberries ru, www widberries, dei sx, ru co,...",[Widberries Вряд придет www widberries ru seer...
6,5,25,5_12 000_000_20 000_04,gedjt8pj скидка 20,"[12 000, 000, 20 000, 04, 35 000, ozon preiu, ...",[Ура Ozon Trve 7 апреля отметил день рождения ...
7,6,24,6_gedenidze_gedenidze reuers_irki gedenidze_irki,внезапно скончался информатор,"[gedenidze, gedenidze reuers, irki gedenidze, ...",[Фото Aexey Bekin news ru Gob Look Press Север...
8,7,20,7_teer_reuers teer_teer msh_teer coonecssd,ведомстве уточнили истребители,"[teer, reuers teer, teer msh, teer coonecssd, ...",[Фото Oeksndr Rushnik Reuers Системы противово...
9,8,20,8_1941_1945_1950_1942,французские военные впервые,"[1941, 1945, 1950, 1942, 1 1, 1944, cu, mrve, ...",[Фото Jerey Moeer Gey Ies Камуфляж возвращаетс...


In [25]:
topic_model.get_topic_info().to_csv(f'../data/interim/dumptopics-{representation_model_name}-{embedding_model_name}-{ngram_range}-HDBScan5.csv', index=False)

In [56]:
topic_model.topic_embeddings_.shape

(29, 312)

In [57]:
topic_model.visualize_topics()

In [58]:
topic_model.visualize_barchart(top_n_topics=30, n_words=5, title='Топ слов по темам', width=400, height=250, custom_labels=True)

---

# Оценка качества

In [60]:

    
print("📊 Выполнение комплексной оценки...")

# Создаем оценщик
evaluator = TopicModelEvaluator(
    topic_model=topic_model,
    texts=to_proccess_text[:100],
    topics=topics,
    embeddings=embeddings
)

# Выполняем комплексную оценку
results = evaluator.evaluate_comprehensive()

# Генерируем отчет
report = evaluator.generate_evaluation_report(
save_path="../data/interim/topic_evaluation_report.csv"
)

print("\n📈 Результаты оценки:")
print("=" * 50)

for metric, info in report.iterrows():
    print(f"{metric:30s}: {info['Value']:10.4f} - {info['Interpretation']}")

# Дополнительная оценка CV Coherence
coherence_results = evaluator.calculate_cv_coherence()
print(f"\n🎯 Средняя CV Coherence тем: {coherence_results.get('avg_cv_coherence', 0):.4f}")


📊 Выполнение комплексной оценки...


Ошибка при вычислении когерентности для темы 2: unable to interpret topic as either a list of tokens or a list of ids
Ошибка при вычислении когерентности для темы 3: unable to interpret topic as either a list of tokens or a list of ids
Ошибка при вычислении когерентности для темы 4: unable to interpret topic as either a list of tokens or a list of ids
Ошибка при вычислении когерентности для темы 7: unable to interpret topic as either a list of tokens or a list of ids
Ошибка при вычислении когерентности для темы 9: unable to interpret topic as either a list of tokens or a list of ids
Ошибка при вычислении когерентности для темы 12: unable to interpret topic as either a list of tokens or a list of ids
Ошибка при вычислении когерентности для темы 13: unable to interpret topic as either a list of tokens or a list of ids
Ошибка при вычислении когерентности для темы 14: unable to interpret topic as either a list of tokens or a list of ids
Ошибка при вычислении когерентности для темы 17: unab


📈 Результаты оценки:
n_topics                      :    28.0000 - Количество обнаруженных тем
outlier_ratio                 :     0.1753 - Доля документов-выбросов (чем меньше, тем лучше)
avg_topic_size                :    26.2143 - Специфическая метрика
std_topic_size                :    30.7099 - Специфическая метрика
silhouette_score              :     0.0588 - Качество кластеризации (-1 до 1, выше лучше)
calinski_harabasz_score       :    13.7684 - Отношение межгрупповой/внутригрупповой дисперсии (выше лучше)
davies_bouldin_score          :     2.4647 - Средняя схожесть кластеров (ниже лучше)
avg_topic_representation_length:    10.0000 - Специфическая метрика
std_topic_representation_length:     0.0000 - Специфическая метрика
topic_word_uniqueness         :     0.9571 - Уникальность слов в темах (выше лучше)
topic_entropy                 :     4.1434 - Энтропия распределения тем (выше = более равномерно)
topic_gini_coefficient        :     0.4852 - Неравномерность распределения (0

Ошибка при вычислении когерентности для темы 2: unable to interpret topic as either a list of tokens or a list of ids
Ошибка при вычислении когерентности для темы 3: unable to interpret topic as either a list of tokens or a list of ids
Ошибка при вычислении когерентности для темы 4: unable to interpret topic as either a list of tokens or a list of ids
Ошибка при вычислении когерентности для темы 7: unable to interpret topic as either a list of tokens or a list of ids


KeyboardInterrupt: 

## Отрисовка кластеров

In [68]:
# Визуализация кластеров новостей в 2D пространстве с помощью plotly
import plotly.express as px
import pandas as pd
import numpy as np

# Получаем координаты документов в 2D пространстве
embeddings_2d = zipped_embs

# Получаем данные о документах
doc_info = topic_model.get_document_info(data['cleaned_text'].tolist())

# Создаем DataFrame для визуализации
plot_df = pd.DataFrame({
    'x': embeddings_2d.embedding_x,
    'y': embeddings_2d.embedding_y,
    'topic': embeddings_2d.topic,
    'text': doc_info.Document,
    'topic_name': doc_info.Name
})

# Создаем цветовую схему
colors = px.colors.qualitative.Plotly

# Создаем интерактивную визуализацию
fig = px.scatter(
    plot_df, 
    x='x', 
    y='y', 
    color='topic_name',
    hover_data=['text'],
    title='Кластеризация новостей по темам',
    color_discrete_sequence=colors,
    opacity=0.7,
    size_max=10
)

# Настраиваем внешний вид графика
fig.update_traces(marker=dict(size=8, line=dict(width=1, color='DarkSlateGrey')))
fig.update_layout(
    legend_title_text='Темы',
    xaxis_title="",
    yaxis_title="",
    xaxis=dict(showticklabels=False),
    yaxis=dict(showticklabels=False),
    plot_bgcolor='white'
)

# Отображаем график
fig.show()



AttributeError: 'numpy.ndarray' object has no attribute 'embedding_x'